In [ ]:
pip install requests

In [ ]:
import requests
import csv
import re
import time

# Constants and Configuration
POST_LIMIT = 800  # Number of posts to fetch
BATCH_SIZE = 100  # Number of posts per API request
GENDER_API_URL = "https://gender-api.com/get"
GENDER_API_KEY = "db4ce7396b03d5b87707ed5e6f30b9ea175c683a42fe678edc5960908a764e35"

# Authentication
session_url = "https://bsky.social/xrpc/com.atproto.server.createSession"
credentials = {
    "identifier": "bushra7472.bsky.social",
    "password": "password12345"
}
response = requests.post(session_url, json=credentials)
if response.status_code != 200:
    print("Authentication failed:", response.status_code, response.text)
    exit()
access_token = response.json().get("accessJwt")
headers = {"Authorization": f"Bearer {access_token}"}

# Gender inference cache
gender_cache = {}

def clean_name(name):
    """Clean a name by removing special characters and whitespace."""
    return re.sub(r'[^\w\s]', '', name).strip()

def infer_gender(display_name, label):
    """Infer gender using displayName or label."""
    candidates = [display_name, label]
    for name in candidates:
        if not name:
            continue
        cleaned_name = clean_name(name)
        if cleaned_name in gender_cache:  # Use cached result if available
            return gender_cache[cleaned_name]
        if cleaned_name:
            first_name = cleaned_name.split()[0]  # Use only the first name
            params = {"name": first_name, "key": GENDER_API_KEY}
            try:
                response = requests.get(GENDER_API_URL, params=params, timeout=10)
                if response.status_code == 200:
                    gender = response.json().get("gender", "unknown")
                    gender_cache[cleaned_name] = gender
                    return gender
            except Exception as e:
                print(f"Error fetching gender for {cleaned_name}: {e}")
    return "unknown"

def fetch_feed():
    """Fetch posts with pagination and show progress."""
    feed = []
    cursor = None
    while len(feed) < POST_LIMIT:
        params = {"limit": BATCH_SIZE}
        if cursor:
            params["cursor"] = cursor
        response = requests.get("https://bsky.social/xrpc/app.bsky.feed.getTimeline", headers=headers, params=params)
        if response.status_code != 200:
            print("Failed to fetch feed:", response.status_code, response.text)
            break
        data = response.json()
        new_posts = data.get("feed", [])
        feed += new_posts
        cursor = data.get("cursor")
        print(f"Fetched {len(feed)} posts so far...")  # Progress message
        if not cursor:
            break
    return feed[:POST_LIMIT]

def fetch_likes(post_uri):
    """Fetch likes for a given post."""
    url = "https://bsky.social/xrpc/app.bsky.feed.getLikes"
    params = {"uri": post_uri}
    response = requests.get(url, headers=headers, params=params)
    return response.json().get("likes", []) if response.status_code == 200 else []

def fetch_followers(actor):
    """Fetch followers of a given user."""
    url = "https://bsky.social/xrpc/app.bsky.graph.getFollowers"
    params = {"actor": actor, "limit": 100}
    response = requests.get(url, headers=headers, params=params)
    return response.json().get("followers", []) if response.status_code == 200 else []

# Main Data Collection
nodes = {}
edges = []
feed = fetch_feed()

print(f"Fetched {len(feed)} posts in total.")
for item in feed:
    post = item.get("post", {})
    author_handle = post.get("author", {}).get("handle")
    display_name = post.get("author", {}).get("displayName", author_handle)
    label = author_handle.replace(".bsky.social", "")
    profile_url = f"https://bsky.app/profile/{author_handle}"

    if author_handle and author_handle not in nodes:
        nodes[author_handle] = {
            "displayName": display_name,
            "label": label,
            "gender": infer_gender(display_name, label),
            "profileURL": profile_url
        }

    for like in fetch_likes(post.get("uri")):
        liker_handle = like.get("actor", {}).get("handle")
        liker_display_name = like.get("actor", {}).get("displayName", liker_handle)
        liker_label = liker_handle.replace(".bsky.social", "") if liker_handle else ""
        liker_url = f"https://bsky.app/profile/{liker_handle}" if liker_handle else ""

        if liker_handle:
            edges.append((liker_handle, author_handle, "like"))
            if liker_handle not in nodes:
                nodes[liker_handle] = {
                    "displayName": liker_display_name,
                    "label": liker_label,
                    "gender": infer_gender(liker_display_name, liker_label),
                    "profileURL": liker_url
                }

    for follower in fetch_followers(author_handle):
        follower_handle = follower.get("handle")
        follower_display_name = follower.get("displayName", follower_handle)
        follower_label = follower_handle.replace(".bsky.social", "") if follower_handle else ""
        follower_url = f"https://bsky.app/profile/{follower_handle}" if follower_handle else ""

        if follower_handle:
            edges.append((follower_handle, author_handle, "follows"))
            if follower_handle not in nodes:
                nodes[follower_handle] = {
                    "displayName": follower_display_name,
                    "label": follower_label,
                    "gender": infer_gender(follower_display_name, follower_label),
                    "profileURL": follower_url
                }

# Save to CSV
with open("nodes.csv", "w", newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["Id", "Label", "DisplayName", "Gender", "ProfileURL"])
    for node_id, data in nodes.items():
        writer.writerow([node_id, data["label"], data["displayName"], data["gender"], data["profileURL"]])

with open("edges.csv", "w", newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["Source", "Target", "Interaction"])
    for edge in edges:
        writer.writerow(edge)

print("Data saved as 'nodes.csv' and 'edges.csv'.")


Fetched 85 posts so far...
Fetched 180 posts so far...
Fetched 239 posts so far...
Fetched 297 posts so far...
Fetched 380 posts so far...
Fetched 452 posts so far...
Fetched 512 posts so far...
Fetched 588 posts so far...
Fetched 636 posts so far...
Fetched 675 posts so far...
Fetched 770 posts so far...
Fetched 812 posts so far...
Fetched 800 posts in total.
Data saved as 'nodes.csv' and 'edges.csv'.


In [ ]:
print(f"Total posts fetched: {len(feed)}")
print(f"Total nodes so far: {len(nodes)}")
print(f"Total edges so far: {len(edges)}")


Total posts fetched: 800
Total nodes so far: 13763
Total edges so far: 49042
